# Phase 5: Targeted Analysis 8: Universal Core Mechanism Profiling

## Overview

NB01 established that the universal core (edges shared across all 5 frequency bands) is **partially sufficient** (29-62% accuracy, 868-4970x above random-K) while band-specific edges alone achieve ~0%. But we never asked: **what does the universal core compute?**

This notebook cross-references structural membership (Phase 2) with head functional roles (Phase 3) to build a mechanistic profile of the universal core.

## Key Questions

1. **Head role composition**: Are induction or previous-token heads overrepresented in the universal core?
2. **Enrichment**: Which head roles are statistically enriched among universal heads (Fisher exact test)?
3. **Layer profile**: Where are universal edges concentrated: early, middle, or late layers?
4. **Band-specific stability**: Are band-specific edges draw-unstable (ACDC noise) or draw-stable (genuine signal)?
5. **Edge type taxonomy**: Do universal and non-universal edges differ in their edge categories (attn-to-attn, mlp-to-attn, etc.)?

## Hypotheses

- H1: Induction heads are enriched in the universal core (they implement the frequency-invariant token-matching algorithm)
- H2: Universal edges are concentrated in middle layers (computation, not I/O)
- H3: Band-specific edges are mostly draw-unstable (ACDC discovery noise)
- H4: Universal edges have more attn-to-attn connections (attention composition)

## Sections

1. Setup & Data Loading
2. Head Role x Universality Cross-Reference
3. Enrichment Analysis (Fisher Exact Test)
4. Layer-Level Mechanism Profile
5. Band-Specific Edge Draw Stability
6. Edge Type Taxonomy
7. Visualizations
8. Summary

## Data Sources

- Phase 3: '04_head_role_classification.csv', '04_copy_scores.csv'
- Phase 2: 'deep_head_universality.csv', 'deep_edge_sharing_raw.csv', 'deep_draw_stability.csv', 'deep_layer_universal_fraction.csv', 'deep_sharing_by_component.csv', 'deep_reliable_band_specific.csv'

**GPU Required**: NO (CPU only, pure data analysis)

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats as sp_stats
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

# Paths
ANALYSIS_ROOT = Path("LSC_circuit_analysis")
PHASE2_DIR = ANALYSIS_ROOT / "02_Phase_Structural" / "outputs" / "analysis"
PHASE3_ATTN = (
    ANALYSIS_ROOT
    / "03_Phase_Representational"
    / "outputs"
    / "attention"
    / "base"
    / "analysis"
)
PHASE5_DIR = ANALYSIS_ROOT / "05_Phase_Targeted"
OUT_ANALYSIS = PHASE5_DIR / "outputs" / "analysis"
OUT_VIZ = PHASE5_DIR / "outputs" / "viz"
OUT_ANALYSIS.mkdir(parents=True, exist_ok=True)
OUT_VIZ.mkdir(parents=True, exist_ok=True)

MODELS = ["pythia-70m", "pythia-160m", "pythia-410m", "pythia-1b", "pythia-1.4b"]
BANDS = ["low", "medium", "high", "very_high", "control"]
ROLE_ORDER = ["induction", "previous_token", "bos_sink", "diffuse"]
ROLE_COLORS = {
    "induction": "#E53935",
    "previous_token": "#FB8C00",
    "bos_sink": "#1E88E5",
    "diffuse": "#9E9E9E",
}

sns.set_theme(style="whitegrid", font_scale=1.1)


def save_figure(fig, filename):
    path = OUT_VIZ / filename
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {path}")


print("Setup complete.")

Setup complete.


In [2]:
# Load all input CSVs

# Phase 3: head roles and copy scores
df_roles = pd.read_csv(PHASE3_ATTN / "04_head_role_classification.csv")
df_copy = pd.read_csv(PHASE3_ATTN / "04_copy_scores.csv")
print(f"Head roles: {len(df_roles)} rows, columns: {list(df_roles.columns)}")
print(f"Copy scores: {len(df_copy)} rows")

# Phase 2: structural data
df_head_univ = pd.read_csv(PHASE2_DIR / "deep_head_universality.csv")
df_edge_sharing = pd.read_csv(PHASE2_DIR / "deep_edge_sharing_raw.csv")
df_draw_stab = pd.read_csv(PHASE2_DIR / "deep_draw_stability.csv")
df_layer_univ = pd.read_csv(PHASE2_DIR / "deep_layer_universal_fraction.csv")
df_sharing_comp = pd.read_csv(PHASE2_DIR / "deep_sharing_by_component.csv")
df_reliable_bs = pd.read_csv(PHASE2_DIR / "deep_reliable_band_specific.csv")

print(f"Head universality: {len(df_head_univ)} rows")
print(f"Edge sharing raw: {len(df_edge_sharing)} rows")
print(f"Draw stability: {len(df_draw_stab)} rows")
print(f"Layer universal fraction: {len(df_layer_univ)} rows")
print(f"Sharing by component: {len(df_sharing_comp)} rows")
print(f"Reliable band-specific: {len(df_reliable_bs)} rows")

print("\nAll data loaded.")

Head roles: 1088 rows, columns: ['layer', 'head', 'role', 'bos_fraction', 'self_fraction', 'prev_fraction', 'source_fraction', 'target_fraction', 'repeat_fraction', 'induction_score', 'distractor_fraction', 'entropy', 'model']
Copy scores: 1088 rows


Head universality: 3264 rows
Edge sharing raw: 50735 rows
Draw stability: 68610 rows
Layer universal fraction: 246 rows
Sharing by component: 75 rows
Reliable band-specific: 438 rows

All data loaded.


## 2. Head Role x Universality Cross-Reference

Merge head role classification with head universality scores. Classify heads as "universal" (universality_score >= 0.8, i.e., edges in >= 4/5 bands) vs "non-universal".

In [3]:
# Average universality across draws to get one value per head per model
df_head_univ_avg = (
    df_head_univ.groupby(["model", "layer", "head_idx"])
    .agg(
        mean_universality=("universality_score", "mean"),
        mean_n_bands=("n_bands", "mean"),
    )
    .reset_index()
)

# Merge with roles on (model, layer, head)
# head_role_classification uses 'head' as integer index = head_idx in universality
df_merged = df_roles.merge(
    df_head_univ_avg,
    left_on=["model", "layer", "head"],
    right_on=["model", "layer", "head_idx"],
    how="inner",
)

# Merge copy scores
df_merged = df_merged.merge(df_copy, on=["model", "layer", "head"], how="left")

# Classify: universal vs non-universal
df_merged["is_universal"] = df_merged["mean_universality"] >= 0.8
df_merged["universality_class"] = df_merged["is_universal"].map(
    {True: "universal", False: "non-universal"}
)

print(f"Merged dataset: {len(df_merged)} heads")
print(f"\nUniversality distribution:")
for model in MODELS:
    sub = df_merged[df_merged["model"] == model]
    n_univ = sub["is_universal"].sum()
    n_total = len(sub)
    print(f"  {model}: {n_univ}/{n_total} universal ({n_univ / n_total:.1%})")

print(f"\nRole distribution by universality class:")
role_univ = (
    df_merged.groupby(["universality_class", "role"]).size().unstack(fill_value=0)
)
print(role_univ)

# Save
df_merged.to_csv(OUT_ANALYSIS / "head_role_universality.csv", index=False)
print(f"\nSaved: {OUT_ANALYSIS / 'head_role_universality.csv'}")

Merged dataset: 1088 heads

Universality distribution:
  pythia-70m: 36/48 universal (75.0%)
  pythia-160m: 80/144 universal (55.6%)
  pythia-410m: 159/384 universal (41.4%)
  pythia-1b: 58/128 universal (45.3%)
  pythia-1.4b: 119/384 universal (31.0%)

Role distribution by universality class:
role                bos_sink  diffuse  induction  previous_token
universality_class                                              
non-universal            480      141          1              14
universal                190      167         26              69

Saved: LSC_circuit_analysis/05_Phase_Targeted/outputs/analysis/head_role_universality.csv


In [4]:
# Per-model role composition breakdown
print("=" * 80)
print("HEAD ROLE COMPOSITION: Universal vs Non-Universal")
print("=" * 80)

for model in MODELS:
    sub = df_merged[df_merged["model"] == model]
    print(f"\n--- {model} ---")
    for uclass in ["universal", "non-universal"]:
        csub = sub[sub["universality_class"] == uclass]
        n = len(csub)
        print(f"  {uclass} ({n} heads):")
        for role in ROLE_ORDER:
            count = (csub["role"] == role).sum()
            pct = count / n * 100 if n > 0 else 0
            print(f"    {role:<16s}: {count:3d} ({pct:5.1f}%)")

    # Also show mean induction score and copy score for universal heads
    univ = sub[sub["is_universal"]]
    non_univ = sub[~sub["is_universal"]]
    print(
        f"  Mean induction score:  universal={univ['induction_score'].mean():.4f}, "
        f"non-universal={non_univ['induction_score'].mean():.4f}"
    )
    print(
        f"  Mean copy score:       universal={univ['copy_score'].mean():.4f}, "
        f"non-universal={non_univ['copy_score'].mean():.4f}"
    )
    print(
        f"  Mean entropy:          universal={univ['entropy'].mean():.4f}, "
        f"non-universal={non_univ['entropy'].mean():.4f}"
    )

HEAD ROLE COMPOSITION: Universal vs Non-Universal

--- pythia-70m ---
  universal (36 heads):
    induction      :  3 (  8.3%)
    previous_token : 11 ( 30.6%)
    bos_sink       :  8 ( 22.2%)
    diffuse        : 14 ( 38.9%)
  non-universal (12 heads):
    induction      :  0 (  0.0%)
    previous_token :  0 (  0.0%)
    bos_sink       :  9 ( 75.0%)
    diffuse        :  3 ( 25.0%)
  Mean induction score:  universal=0.0654, non-universal=0.0139
  Mean copy score:       universal=0.0094, non-universal=0.0078
  Mean entropy:          universal=2.7323, non-universal=2.4368

--- pythia-160m ---
  universal (80 heads):
    induction      :  5 (  6.2%)
    previous_token : 17 ( 21.2%)
    bos_sink       : 33 ( 41.2%)
    diffuse        : 25 ( 31.2%)
  non-universal (64 heads):
    induction      :  0 (  0.0%)
    previous_token :  3 (  4.7%)
    bos_sink       : 49 ( 76.6%)
    diffuse        : 12 ( 18.8%)
  Mean induction score:  universal=0.0444, non-universal=0.0180
  Mean copy score:   

## 3. Enrichment Analysis (Fisher Exact Test)

For each head role x model, compute a 2x2 contingency table:

|              | Universal | Non-universal |
|--------------|-----------|---------------|
| Role         |     a     |       b       |
| Not-role     |     c     |       d       |

Fisher exact test -> odds ratio + p-value. Odds ratio > 1 means the role is enriched among universal heads.

In [5]:
enrichment_rows = []

for model in MODELS:
    sub = df_merged[df_merged["model"] == model]
    n_universal = sub["is_universal"].sum()
    n_non_universal = (~sub["is_universal"]).sum()

    for role in ROLE_ORDER:
        # 2x2 contingency table
        a = ((sub["role"] == role) & sub["is_universal"]).sum()  # role + universal
        b = ((sub["role"] == role) & ~sub["is_universal"]).sum()  # role + non-universal
        c = ((sub["role"] != role) & sub["is_universal"]).sum()  # not-role + universal
        d = (
            (sub["role"] != role) & ~sub["is_universal"]
        ).sum()  # not-role + non-universal

        table = np.array([[a, b], [c, d]])
        odds_ratio, p_value = sp_stats.fisher_exact(table, alternative="two-sided")

        # Compute 95% CI for log(OR) using Woolf's method
        # log(OR) ~ N(log(OR), SE) where SE = sqrt(1/a + 1/b + 1/c + 1/d)
        cells = [a, b, c, d]
        if all(x > 0 for x in cells):
            log_or = np.log(odds_ratio)
            se = np.sqrt(sum(1 / x for x in cells))
            ci_low = np.exp(log_or - 1.96 * se)
            ci_high = np.exp(log_or + 1.96 * se)
        else:
            ci_low = ci_high = np.nan

        # Role fraction in universal vs non-universal
        frac_univ = a / n_universal if n_universal > 0 else 0
        frac_non_univ = b / n_non_universal if n_non_universal > 0 else 0

        enrichment_rows.append(
            {
                "model": model,
                "role": role,
                "n_role_universal": int(a),
                "n_role_non_universal": int(b),
                "n_not_role_universal": int(c),
                "n_not_role_non_universal": int(d),
                "frac_in_universal": frac_univ,
                "frac_in_non_universal": frac_non_univ,
                "odds_ratio": odds_ratio,
                "ci_low": ci_low,
                "ci_high": ci_high,
                "p_value": p_value,
                "significant": p_value < 0.05,
            }
        )

df_enrichment = pd.DataFrame(enrichment_rows)
df_enrichment.to_csv(OUT_ANALYSIS / "role_universality_enrichment.csv", index=False)

print("=" * 80)
print("ENRICHMENT ANALYSIS: Head Roles in Universal Core")
print("=" * 80)
print(
    f"{'Model':<14s} {'Role':<18s} {'OR':>6s} {'95% CI':>16s} {'p-value':>10s} {'Sig':>4s} {'Univ%':>7s} {'Non%':>7s}"
)
print("-" * 80)

for _, row in df_enrichment.iterrows():
    sig = "*" if row["significant"] else " "
    ci_str = (
        f"[{row['ci_low']:.2f}, {row['ci_high']:.2f}]"
        if not np.isnan(row["ci_low"])
        else "[nan]"
    )
    print(
        f"{row['model']:<14s} {row['role']:<18s} {row['odds_ratio']:>6.2f} {ci_str:>16s} "
        f"{row['p_value']:>10.4f} {sig:>4s} {row['frac_in_universal']:>6.1%} {row['frac_in_non_universal']:>6.1%}"
    )

print(f"\nSaved: {OUT_ANALYSIS / 'role_universality_enrichment.csv'}")

ENRICHMENT ANALYSIS: Head Roles in Universal Core
Model          Role                   OR           95% CI    p-value  Sig   Univ%    Non%
--------------------------------------------------------------------------------
pythia-70m     induction             inf            [nan]     0.5629        8.3%   0.0%
pythia-70m     previous_token        inf            [nan]     0.0442    *  30.6%   0.0%
pythia-70m     bos_sink             0.10     [0.02, 0.44]     0.0017    *  22.2%  75.0%
pythia-70m     diffuse              1.91     [0.44, 8.29]     0.4969       38.9%  25.0%
pythia-160m    induction             inf            [nan]     0.0658        6.2%   0.0%
pythia-160m    previous_token       5.49    [1.53, 19.67]     0.0064    *  21.2%   4.7%
pythia-160m    bos_sink             0.21     [0.10, 0.45]     0.0000    *  41.2%  76.6%
pythia-160m    diffuse              1.97     [0.90, 4.32]     0.1242       31.2%  18.8%
pythia-410m    induction           10.32    [1.26, 84.70]     0.0100    *  

## 4. Layer-Level Mechanism Profile

Where are universal edges concentrated? Cross-reference per-layer universal fraction with dominant head roles.

In [6]:
# Average layer universal fraction across draws
df_layer_avg = (
    df_layer_univ.groupby(["model", "layer"])
    .agg(
        total_edges=("total_edges", "mean"),
        universal_count=("universal_count", "mean"),
        universal_fraction=("universal_fraction", "mean"),
    )
    .reset_index()
)

# Get dominant head role at each layer
layer_roles = (
    df_merged.groupby(["model", "layer", "role"]).size().reset_index(name="count")
)
layer_dominant = layer_roles.loc[
    layer_roles.groupby(["model", "layer"])["count"].idxmax()
][["model", "layer", "role"]].rename(columns={"role": "dominant_role"})

# Also get per-layer universal head fraction
layer_head_univ = (
    df_merged.groupby(["model", "layer"])
    .agg(
        n_heads=("head", "count"),
        n_universal_heads=("is_universal", "sum"),
        mean_induction_score=("induction_score", "mean"),
    )
    .reset_index()
)
layer_head_univ["universal_head_fraction"] = (
    layer_head_univ["n_universal_heads"] / layer_head_univ["n_heads"]
)

# Merge
df_layer_profile = df_layer_avg.merge(layer_dominant, on=["model", "layer"], how="left")
df_layer_profile = df_layer_profile.merge(
    layer_head_univ, on=["model", "layer"], how="left"
)


# Classify layer position
def classify_layer_position(row):
    n_layers = {
        "pythia-70m": 6,
        "pythia-160m": 12,
        "pythia-410m": 24,
        "pythia-1b": 16,
        "pythia-1.4b": 24,
    }
    total = n_layers.get(row["model"], 12)
    frac = row["layer"] / total
    if frac < 0.33:
        return "early"
    elif frac < 0.67:
        return "middle"
    else:
        return "late"


df_layer_profile["position"] = df_layer_profile.apply(classify_layer_position, axis=1)

df_layer_profile.to_csv(OUT_ANALYSIS / "layer_mechanism_profile.csv", index=False)

# Summary: where are universal edges concentrated?
print("=" * 80)
print("LAYER MECHANISM PROFILE")
print("=" * 80)

for model in MODELS:
    sub = df_layer_profile[df_layer_profile["model"] == model]
    print(f"\n--- {model} ({len(sub)} layers) ---")

    # Position summary
    for pos in ["early", "middle", "late"]:
        psub = sub[sub["position"] == pos]
        if len(psub) > 0:
            mean_frac = psub["universal_fraction"].mean()
            mean_edges = psub["universal_count"].mean()
            total_edges = psub["universal_count"].sum()
            print(
                f"  {pos:>6s}: {mean_frac:.1%} universal fraction, "
                f"{total_edges:.0f} universal edges total"
            )

    # Top layers by universal count
    top = sub.nlargest(3, "universal_count")
    print(f"  Top layers: ", end="")
    for _, r in top.iterrows():
        print(
            f"L{int(r['layer'])}({r['universal_count']:.0f} edges, {r['dominant_role']})",
            end="  ",
        )
    print()

print(f"\nSaved: {OUT_ANALYSIS / 'layer_mechanism_profile.csv'}")

LAYER MECHANISM PROFILE

--- pythia-70m (6 layers) ---
   early: 79.3% universal fraction, 31 universal edges total
  middle: 47.6% universal fraction, 170 universal edges total
    late: 68.5% universal fraction, 101 universal edges total
  Top layers: L5(101 edges, bos_sink)  L3(94 edges, diffuse)  L4(41 edges, bos_sink)  

--- pythia-160m (12 layers) ---
   early: 30.4% universal fraction, 45 universal edges total
  middle: 26.0% universal fraction, 326 universal edges total
    late: 39.5% universal fraction, 342 universal edges total
  Top layers: L11(185 edges, bos_sink)  L5(95 edges, bos_sink)  L9(92 edges, bos_sink)  

--- pythia-410m (24 layers) ---
   early: 12.9% universal fraction, 86 universal edges total
  middle: 15.0% universal fraction, 564 universal edges total
    late: 21.7% universal fraction, 653 universal edges total
  Top layers: L23(323 edges, bos_sink)  L16(96 edges, bos_sink)  L11(95 edges, bos_sink)  

--- pythia-1b (16 layers) ---
   early: 24.3% universal 

## 5. Band-Specific Edge Draw Stability

Are band-specific edges draw-stable (genuine signal) or draw-unstable (ACDC noise)? Compare draw stability across sharing levels.

In [7]:
# Get sharing_level per edge from edge_sharing_raw
# Average sharing_level across draws for each edge
edge_sharing_avg = (
    df_edge_sharing.groupby(["raw", "model"])
    .agg(
        mean_sharing_level=("sharing_level", "mean"),
    )
    .reset_index()
)


# Classify sharing category
def sharing_category(sl):
    if sl >= 4.5:  # sharing_level 5 = universal
        return "universal (5)"
    elif sl >= 1.5:  # sharing_level 2-4 = shared
        return "shared (2-4)"
    else:  # sharing_level 1 = band-specific
        return "band-specific (1)"


edge_sharing_avg["sharing_cat"] = edge_sharing_avg["mean_sharing_level"].apply(
    sharing_category
)

# Merge with draw stability
# draw_stability has (raw, model, band, n_draws): one row per edge per band
df_stab_merged = df_draw_stab.merge(edge_sharing_avg, on=["raw", "model"], how="left")

print(f"Merged stability data: {len(df_stab_merged)} rows")
print(f"Sharing categories: {df_stab_merged['sharing_cat'].value_counts().to_dict()}")

# For each sharing category: compute fraction of edges that are draw-stable (n_draws=3)
stability_rows = []

for model in MODELS:
    msub = df_stab_merged[df_stab_merged["model"] == model]

    for cat in ["universal (5)", "shared (2-4)", "band-specific (1)"]:
        csub = msub[msub["sharing_cat"] == cat]
        if len(csub) == 0:
            continue

        n_total = len(csub)
        n_stable = (csub["n_draws"] == 3).sum()  # present in all 3 draws
        n_partial = (csub["n_draws"] == 2).sum()  # present in 2/3 draws
        n_unstable = (csub["n_draws"] == 1).sum()  # present in only 1 draw

        stability_rows.append(
            {
                "model": model,
                "sharing_category": cat,
                "n_entries": n_total,
                "n_stable_3": int(n_stable),
                "n_partial_2": int(n_partial),
                "n_unstable_1": int(n_unstable),
                "frac_stable": n_stable / n_total if n_total > 0 else 0,
                "frac_unstable": n_unstable / n_total if n_total > 0 else 0,
                "mean_n_draws": csub["n_draws"].mean(),
            }
        )

# Also compute per-band band-specific stability
for model in MODELS:
    msub = df_stab_merged[
        (df_stab_merged["model"] == model)
        & (df_stab_merged["sharing_cat"] == "band-specific (1)")
    ]
    for band in BANDS:
        bsub = msub[msub["band"] == band]
        if len(bsub) == 0:
            continue
        stability_rows.append(
            {
                "model": model,
                "sharing_category": f"band-specific: {band}",
                "n_entries": len(bsub),
                "n_stable_3": int((bsub["n_draws"] == 3).sum()),
                "n_partial_2": int((bsub["n_draws"] == 2).sum()),
                "n_unstable_1": int((bsub["n_draws"] == 1).sum()),
                "frac_stable": (bsub["n_draws"] == 3).mean(),
                "frac_unstable": (bsub["n_draws"] == 1).mean(),
                "mean_n_draws": bsub["n_draws"].mean(),
            }
        )

df_stability = pd.DataFrame(stability_rows)
df_stability.to_csv(OUT_ANALYSIS / "band_specific_stability.csv", index=False)

print("\n" + "=" * 80)
print("DRAW STABILITY BY SHARING CATEGORY")
print("=" * 80)
print(
    f"{'Model':<14s} {'Category':<22s} {'N':>6s} {'Stable':>8s} {'Partial':>8s} {'Unstable':>8s} {'Mean':>6s}"
)
print("-" * 80)

main_cats = ["universal (5)", "shared (2-4)", "band-specific (1)"]
for model in MODELS:
    for cat in main_cats:
        row = df_stability[
            (df_stability["model"] == model) & (df_stability["sharing_category"] == cat)
        ]
        if len(row) == 0:
            continue
        r = row.iloc[0]
        print(
            f"{r['model']:<14s} {r['sharing_category']:<22s} {r['n_entries']:>6d} "
            f"{r['frac_stable']:>7.1%} {r['n_partial_2'] / r['n_entries']:>7.1%} "
            f"{r['frac_unstable']:>7.1%} {r['mean_n_draws']:>6.2f}"
        )

print(f"\nSaved: {OUT_ANALYSIS / 'band_specific_stability.csv'}")

Merged stability data: 68610 rows
Sharing categories: {'shared (2-4)': 39528, 'universal (5)': 14815, 'band-specific (1)': 14267}



DRAW STABILITY BY SHARING CATEGORY
Model          Category                    N   Stable  Partial Unstable   Mean
--------------------------------------------------------------------------------
pythia-70m     universal (5)            1495   97.9%    2.1%    0.0%   2.98
pythia-70m     shared (2-4)              751   32.2%   33.6%   34.2%   1.98
pythia-70m     band-specific (1)         165    1.2%   17.6%   81.2%   1.20
pythia-160m    universal (5)            3310   95.9%    4.0%    0.0%   2.96
pythia-160m    shared (2-4)             5320   24.2%   35.2%   40.7%   1.83
pythia-160m    band-specific (1)        1306    1.7%   14.7%   83.6%   1.18
pythia-410m    universal (5)            5815   94.8%    5.0%    0.2%   2.95
pythia-410m    shared (2-4)            17323   17.8%   35.0%   47.2%   1.71
pythia-410m    band-specific (1)        6094    1.6%   12.3%   86.1%   1.16
pythia-1b      universal (5)            1580   93.9%    6.1%    0.1%   2.94
pythia-1b      shared (2-4)             4319

## 6. Edge Type Taxonomy

Compare edge category composition (attn-to-attn, mlp-to-attn, etc.) between universal and non-universal edges.

In [8]:
# Edge type taxonomy from edge_sharing_raw
# Classify as universal (sharing_level=5) vs non-universal
df_edge_sharing["is_universal_edge"] = df_edge_sharing["sharing_level"] == 5

# Edge category composition by universality
edge_type_rows = []

for model in MODELS:
    msub = df_edge_sharing[df_edge_sharing["model"] == model]

    for is_univ, label in [(True, "universal"), (False, "non-universal")]:
        usub = msub[msub["is_universal_edge"] == is_univ]
        n_total = len(usub)
        if n_total == 0:
            continue

        # Edge category counts
        cat_counts = usub["edge_category"].value_counts()
        for cat, count in cat_counts.items():
            edge_type_rows.append(
                {
                    "model": model,
                    "universality": label,
                    "edge_category": cat,
                    "count": int(count),
                    "fraction": count / n_total,
                }
            )

        # Skip connection fraction
        if "is_skip" in usub.columns:
            skip_frac = usub["is_skip"].mean()
            edge_type_rows.append(
                {
                    "model": model,
                    "universality": label,
                    "edge_category": "_skip_fraction",
                    "count": int(usub["is_skip"].sum()),
                    "fraction": skip_frac,
                }
            )

df_edge_types = pd.DataFrame(edge_type_rows)

print("=" * 80)
print("EDGE TYPE TAXONOMY: Universal vs Non-Universal")
print("=" * 80)

for model in MODELS:
    print(f"\n--- {model} ---")
    for label in ["universal", "non-universal"]:
        sub = df_edge_types[
            (df_edge_types["model"] == model)
            & (df_edge_types["universality"] == label)
            & (~df_edge_types["edge_category"].str.startswith("_"))
        ]
        total = sub["count"].sum()
        print(f"  {label} ({total} edges):")
        for _, r in sub.sort_values("fraction", ascending=False).iterrows():
            print(
                f"    {r['edge_category']:<20s}: {r['count']:>5d} ({r['fraction']:>5.1%})"
            )

        # Skip fraction
        skip = df_edge_types[
            (df_edge_types["model"] == model)
            & (df_edge_types["universality"] == label)
            & (df_edge_types["edge_category"] == "_skip_fraction")
        ]
        if len(skip) > 0:
            print(
                f"    {'skip connections':<20s}: {skip.iloc[0]['count']:>5d} ({skip.iloc[0]['fraction']:>5.1%})"
            )

EDGE TYPE TAXONOMY: Universal vs Non-Universal

--- pythia-70m ---
  universal (905 edges):
    attn_to_mlp        :  272 (30.1%)
    attn_to_attn       :  249 (27.5%)
    mlp_to_attn        :  145 (16.0%)
    attn_to_resid      :  129 (14.3%)
    mlp_to_mlp         :   45 ( 5.0%)
    embed_to_attn      :   29 ( 3.2%)
    mlp_to_resid       :   18 ( 2.0%)
    embed_to_mlp       :   15 ( 1.7%)
    embed_to_resid     :    3 ( 0.3%)
    skip connections   :  561 (62.0%)
  non-universal (693 edges):
    attn_to_attn       :  444 (64.1%)
    mlp_to_attn        :  155 (22.4%)
    attn_to_mlp        :   67 ( 9.7%)
    embed_to_attn      :   15 ( 2.2%)
    attn_to_resid      :    9 ( 1.3%)
    embed_to_mlp       :    3 ( 0.4%)
    skip connections   :  435 (62.8%)

--- pythia-160m ---
  universal (2140 edges):
    attn_to_mlp        :  764 (35.7%)
    attn_to_attn       :  531 (24.8%)
    attn_to_resid      :  304 (14.2%)
    mlp_to_attn        :  281 (13.1%)
    mlp_to_mlp         :  198 ( 9.

In [9]:
# Build mechanism summary
summary_rows = []

for model in MODELS:
    # Head role composition of universal core
    univ_heads = df_merged[(df_merged["model"] == model) & df_merged["is_universal"]]
    all_heads = df_merged[df_merged["model"] == model]

    dominant_role = univ_heads["role"].mode().iloc[0] if len(univ_heads) > 0 else "N/A"
    n_induction_univ = (univ_heads["role"] == "induction").sum()
    n_induction_total = (all_heads["role"] == "induction").sum()

    # Layer concentration
    lp = df_layer_profile[df_layer_profile["model"] == model]
    position_edges = lp.groupby("position")["universal_count"].sum()
    total_univ_edges = position_edges.sum()
    peak_position = position_edges.idxmax() if len(position_edges) > 0 else "N/A"

    # Stability
    stab = df_stability[
        (df_stability["model"] == model)
        & (df_stability["sharing_category"] == "band-specific (1)")
    ]
    bs_stable_frac = stab.iloc[0]["frac_stable"] if len(stab) > 0 else np.nan

    stab_univ = df_stability[
        (df_stability["model"] == model)
        & (df_stability["sharing_category"] == "universal (5)")
    ]
    univ_stable_frac = (
        stab_univ.iloc[0]["frac_stable"] if len(stab_univ) > 0 else np.nan
    )

    # Enrichment
    enr = df_enrichment[
        (df_enrichment["model"] == model) & (df_enrichment["role"] == "induction")
    ]
    induction_or = enr.iloc[0]["odds_ratio"] if len(enr) > 0 else np.nan
    induction_p = enr.iloc[0]["p_value"] if len(enr) > 0 else np.nan

    # Edge types
    et_univ = df_edge_types[
        (df_edge_types["model"] == model)
        & (df_edge_types["universality"] == "universal")
        & (~df_edge_types["edge_category"].str.startswith("_"))
    ]
    dominant_edge_type = (
        et_univ.loc[et_univ["fraction"].idxmax(), "edge_category"]
        if len(et_univ) > 0
        else "N/A"
    )

    summary_rows.append(
        {
            "model": model,
            "n_universal_heads": len(univ_heads),
            "n_total_heads": len(all_heads),
            "universal_head_fraction": len(univ_heads) / len(all_heads)
            if len(all_heads) > 0
            else 0,
            "dominant_role_in_universal": dominant_role,
            "n_induction_in_universal": int(n_induction_univ),
            "n_induction_total": int(n_induction_total),
            "induction_enrichment_OR": induction_or,
            "induction_enrichment_p": induction_p,
            "peak_layer_position": peak_position,
            "universal_stable_frac": univ_stable_frac,
            "band_specific_stable_frac": bs_stable_frac,
            "dominant_edge_type": dominant_edge_type,
        }
    )

df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(OUT_ANALYSIS / "mechanism_summary.csv", index=False)
print(f"Saved: {OUT_ANALYSIS / 'mechanism_summary.csv'}")
print()
print(df_summary.to_string(index=False))

Saved: LSC_circuit_analysis/05_Phase_Targeted/outputs/analysis/mechanism_summary.csv

      model  n_universal_heads  n_total_heads  universal_head_fraction dominant_role_in_universal  n_induction_in_universal  n_induction_total  induction_enrichment_OR  induction_enrichment_p peak_layer_position  universal_stable_frac  band_specific_stable_frac dominant_edge_type
 pythia-70m                 36             48                 0.750000                    diffuse                         3                  3                      inf                0.562905              middle               0.979264                   0.012121        attn_to_mlp
pythia-160m                 80            144                 0.555556                   bos_sink                         5                  5                      inf                0.065829                late               0.959215                   0.016845        attn_to_mlp
pythia-410m                159            384                 0.414062 

## 7. Visualizations

### VIZ T8_01: Head Role Composition: Universal vs Non-Universal

In [10]:
fig, axes = plt.subplots(1, len(MODELS), figsize=(16, 5), sharey=True)

for ax, model in zip(axes, MODELS):
    sub = df_merged[df_merged["model"] == model]

    # Compute role fractions for universal and non-universal
    data = {}
    for uclass in ["universal", "non-universal"]:
        csub = sub[sub["universality_class"] == uclass]
        n = len(csub)
        fracs = []
        for role in ROLE_ORDER:
            fracs.append((csub["role"] == role).sum() / n if n > 0 else 0)
        data[uclass] = fracs

    x = np.arange(2)
    bottom_univ = 0
    bottom_non = 0

    for i, role in enumerate(ROLE_ORDER):
        heights = [data["universal"][i], data["non-universal"][i]]
        bottoms = [bottom_univ, bottom_non]
        ax.bar(
            x,
            heights,
            bottom=bottoms,
            color=ROLE_COLORS[role],
            label=role if ax == axes[0] else "",
            width=0.6,
            alpha=0.85,
        )
        bottom_univ += heights[0]
        bottom_non += heights[1]

    ax.set_xticks(x)
    n_univ = (sub["universality_class"] == "universal").sum()
    n_non = (sub["universality_class"] == "non-universal").sum()
    ax.set_xticklabels(
        [f"Universal\n(n={n_univ})", f"Non-univ.\n(n={n_non})"], fontsize=9
    )
    ax.set_title(model, fontsize=11, fontweight="bold")
    ax.set_ylim(0, 1.0)

axes[0].set_ylabel("Fraction of Heads")
axes[0].legend(loc="upper left", fontsize=8, title="Role")
fig.suptitle(
    "Head Role Composition: Universal vs Non-Universal Heads",
    fontsize=13,
    fontweight="bold",
)
fig.tight_layout()
save_figure(fig, "T8_01_role_vs_universality.png")

Saved: LSC_circuit_analysis/05_Phase_Targeted/outputs/viz/T8_01_role_vs_universality.png


### VIZ T8_02: Enrichment Odds Ratios (Forest Plot)

In [11]:
fig, axes = plt.subplots(1, len(ROLE_ORDER), figsize=(16, 5), sharey=True)

for ax, role in zip(axes, ROLE_ORDER):
    rsub = df_enrichment[df_enrichment["role"] == role].copy()
    rsub = rsub.set_index("model").reindex(MODELS).reset_index()

    y_pos = np.arange(len(MODELS))

    for i, (_, row) in enumerate(rsub.iterrows()):
        or_val = row["odds_ratio"]
        ci_lo = row["ci_low"]
        ci_hi = row["ci_high"]

        color = ROLE_COLORS[role]
        marker = "o"

        if np.isfinite(or_val) and or_val > 0:
            # Cap for display
            or_display = min(or_val, 50)
            ci_lo_d = max(ci_lo, 0.01) if np.isfinite(ci_lo) else 0.01
            ci_hi_d = min(ci_hi, 50) if np.isfinite(ci_hi) else 50

            ax.errorbar(
                or_display,
                i,
                xerr=[[or_display - ci_lo_d], [ci_hi_d - or_display]],
                fmt=marker,
                color=color,
                markersize=8,
                capsize=4,
                linewidth=1.5,
            )

            if row["significant"]:
                ax.annotate(
                    "*",
                    (or_display, i - 0.3),
                    fontsize=14,
                    fontweight="bold",
                    color="red",
                    ha="center",
                )
        else:
            # Plot at 0 or infinity: use a special marker
            ax.plot(1, i, marker="x", color="gray", markersize=8)

    ax.axvline(x=1, color="black", linestyle="--", linewidth=0.8, alpha=0.5)
    ax.set_xscale("log")
    ax.set_xlim(0.05, 60)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(MODELS)
    ax.set_title(role, fontsize=11, fontweight="bold", color=ROLE_COLORS[role])
    ax.set_xlabel("Odds Ratio")

fig.suptitle(
    "Role Enrichment in Universal Core (Fisher Exact Test)\n"
    "OR > 1 = enriched in universal, * = p < 0.05",
    fontsize=13,
    fontweight="bold",
)
fig.tight_layout()
save_figure(fig, "T8_02_enrichment_odds_ratios.png")

Saved: LSC_circuit_analysis/05_Phase_Targeted/outputs/viz/T8_02_enrichment_odds_ratios.png


### VIZ T8_03: Layer Profile: Universal Edge Fraction by Layer

In [12]:
for model in MODELS:
    sub = df_layer_profile[df_layer_profile["model"] == model].sort_values("layer")

    layers = sub["layer"].values
    fracs = sub["universal_fraction"].values
    roles = sub["dominant_role"].values

    colors = [ROLE_COLORS.get(r, "#9E9E9E") for r in roles]

    n_layers = len(layers)
    fig_width = max(6, n_layers * 0.6)
    fig, ax = plt.subplots(figsize=(fig_width, 5))

    bars = ax.bar(
        layers, fracs, color=colors, alpha=0.85, edgecolor="white", linewidth=0.5
    )

    # Add edge count annotations on top
    for i, (l, f, uc) in enumerate(zip(layers, fracs, sub["universal_count"].values)):
        if uc > 0:
            ax.text(
                l,
                f + 0.02,
                f"{uc:.0f}",
                ha="center",
                va="bottom",
                fontsize=7,
                rotation=90,
            )

    ax.set_xlabel("Layer")
    ax.set_ylabel("Universal Edge Fraction")
    ax.set_title(
        f"{model}: Per-Layer Universal Edge Fraction\n(colored by dominant head role)",
        fontsize=11,
        fontweight="bold",
    )
    ax.set_ylim(0, 1.15)
    ax.set_xticks(layers)

    # Legend for role colors
    from matplotlib.patches import Patch

    legend_elements = [Patch(facecolor=ROLE_COLORS[r], label=r) for r in ROLE_ORDER]
    ax.legend(
        handles=legend_elements, loc="upper right", fontsize=8, title="Dominant Role"
    )

    fig.tight_layout()
    save_figure(fig, f"T8_03_layer_profile_{model}.png")

Saved: LSC_circuit_analysis/05_Phase_Targeted/outputs/viz/T8_03_layer_profile_pythia-70m.png


Saved: LSC_circuit_analysis/05_Phase_Targeted/outputs/viz/T8_03_layer_profile_pythia-160m.png


Saved: LSC_circuit_analysis/05_Phase_Targeted/outputs/viz/T8_03_layer_profile_pythia-410m.png


Saved: LSC_circuit_analysis/05_Phase_Targeted/outputs/viz/T8_03_layer_profile_pythia-1b.png


Saved: LSC_circuit_analysis/05_Phase_Targeted/outputs/viz/T8_03_layer_profile_pythia-1.4b.png


### VIZ T8_04: Draw Stability by Sharing Category

In [13]:
fig, axes = plt.subplots(1, len(MODELS), figsize=(16, 5), sharey=True)

main_cats = ["universal (5)", "shared (2-4)", "band-specific (1)"]
cat_colors = ["#388E3C", "#1976D2", "#F57C00"]

for ax, model in zip(axes, MODELS):
    sub = df_stability[
        (df_stability["model"] == model)
        & (df_stability["sharing_category"].isin(main_cats))
    ]

    x = np.arange(len(main_cats))

    # Stacked bar: stable, partial, unstable
    stable_fracs = []
    partial_fracs = []
    unstable_fracs = []

    for cat in main_cats:
        csub = sub[sub["sharing_category"] == cat]
        if len(csub) > 0:
            r = csub.iloc[0]
            stable_fracs.append(r["frac_stable"])
            partial_fracs.append(
                r["n_partial_2"] / r["n_entries"] if r["n_entries"] > 0 else 0
            )
            unstable_fracs.append(r["frac_unstable"])
        else:
            stable_fracs.append(0)
            partial_fracs.append(0)
            unstable_fracs.append(0)

    ax.bar(
        x,
        stable_fracs,
        color="#388E3C",
        alpha=0.85,
        label="Stable (3/3)" if ax == axes[0] else "",
    )
    ax.bar(
        x,
        partial_fracs,
        bottom=stable_fracs,
        color="#FDD835",
        alpha=0.85,
        label="Partial (2/3)" if ax == axes[0] else "",
    )
    bottom2 = [s + p for s, p in zip(stable_fracs, partial_fracs)]
    ax.bar(
        x,
        unstable_fracs,
        bottom=bottom2,
        color="#E53935",
        alpha=0.85,
        label="Unstable (1/3)" if ax == axes[0] else "",
    )

    ax.set_xticks(x)
    ax.set_xticklabels(["Universal", "Shared", "Band-\nspecific"], fontsize=9)
    ax.set_title(model, fontsize=11, fontweight="bold")
    ax.set_ylim(0, 1.05)

axes[0].set_ylabel("Fraction of Edge-Band Entries")
axes[0].legend(loc="lower left", fontsize=8)
fig.suptitle(
    "Draw Stability: Universal vs Shared vs Band-Specific Edges",
    fontsize=13,
    fontweight="bold",
)
fig.tight_layout()
save_figure(fig, "T8_04_band_specific_stability.png")

Saved: LSC_circuit_analysis/05_Phase_Targeted/outputs/viz/T8_04_band_specific_stability.png


### VIZ T8_05: Edge Type Taxonomy

In [14]:
# Get all edge categories (excluding _skip_fraction)
all_cats = sorted(
    df_edge_types[~df_edge_types["edge_category"].str.startswith("_")][
        "edge_category"
    ].unique()
)

cat_cmap = plt.cm.Set2(np.linspace(0, 1, len(all_cats)))
cat_colors_map = {cat: cat_cmap[i] for i, cat in enumerate(all_cats)}

fig, axes = plt.subplots(1, len(MODELS), figsize=(16, 5), sharey=True)

for ax, model in zip(axes, MODELS):
    for j, label in enumerate(["universal", "non-universal"]):
        sub = df_edge_types[
            (df_edge_types["model"] == model)
            & (df_edge_types["universality"] == label)
            & (~df_edge_types["edge_category"].str.startswith("_"))
        ]

        bottom = 0
        for cat in all_cats:
            csub = sub[sub["edge_category"] == cat]
            frac = csub["fraction"].values[0] if len(csub) > 0 else 0
            ax.bar(
                j,
                frac,
                bottom=bottom,
                color=cat_colors_map[cat],
                label=cat if ax == axes[0] and j == 0 else "",
                width=0.6,
                alpha=0.85,
            )
            bottom += frac

    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Universal", "Non-univ."], fontsize=9)
    ax.set_title(model, fontsize=11, fontweight="bold")
    ax.set_ylim(0, 1.05)

axes[0].set_ylabel("Fraction of Edges")
axes[0].legend(loc="upper left", fontsize=7, title="Edge Type", bbox_to_anchor=(0, 1.0))
fig.suptitle(
    "Edge Category Composition: Universal vs Non-Universal",
    fontsize=13,
    fontweight="bold",
)
fig.tight_layout()
save_figure(fig, "T8_05_edge_type_taxonomy.png")

Saved: LSC_circuit_analysis/05_Phase_Targeted/outputs/viz/T8_05_edge_type_taxonomy.png


## 8. Summary

In [15]:
print("=" * 80)
print("PHASE 5: MECHANISM PROFILING: SUMMARY")
print("=" * 80)

# 1. Dominant role in universal core
print("\n--- Universal Core Dominant Roles ---")
for model in MODELS:
    univ = df_merged[(df_merged["model"] == model) & df_merged["is_universal"]]
    if len(univ) > 0:
        role_dist = univ["role"].value_counts(normalize=True)
        dominant = role_dist.index[0]
        print(
            f"  {model}: {dominant} ({role_dist.iloc[0]:.1%}), "
            f"then {role_dist.index[1]} ({role_dist.iloc[1]:.1%})"
        )

# 2. Enrichment summary
print("\n--- Enrichment Summary ---")
for role in ROLE_ORDER:
    rsub = df_enrichment[df_enrichment["role"] == role]
    n_sig = rsub["significant"].sum()
    mean_or = rsub["odds_ratio"].replace([np.inf, -np.inf], np.nan).mean()
    direction = "ENRICHED" if mean_or > 1 else "DEPLETED"
    print(
        f"  {role:<18s}: mean OR={mean_or:.2f} ({direction}), "
        f"significant in {n_sig}/{len(rsub)} models"
    )

# 3. Layer concentration
print("\n--- Layer Concentration ---")
for model in MODELS:
    lp = df_layer_profile[df_layer_profile["model"] == model]
    pos_edges = lp.groupby("position")["universal_count"].sum()
    total = pos_edges.sum()
    peak = pos_edges.idxmax()
    peak_frac = pos_edges.max() / total if total > 0 else 0
    print(f"  {model}: peak={peak} ({peak_frac:.1%} of universal edges)")

# 4. Draw stability verdict
print("\n--- Draw Stability Verdict ---")
for model in MODELS:
    univ_stab = df_stability[
        (df_stability["model"] == model)
        & (df_stability["sharing_category"] == "universal (5)")
    ]
    bs_stab = df_stability[
        (df_stability["model"] == model)
        & (df_stability["sharing_category"] == "band-specific (1)")
    ]
    if len(univ_stab) > 0 and len(bs_stab) > 0:
        u_frac = univ_stab.iloc[0]["frac_stable"]
        b_frac = bs_stab.iloc[0]["frac_stable"]
        print(
            f"  {model}: universal={u_frac:.1%} stable, "
            f"band-specific={b_frac:.1%} stable"
        )

# 5. Edge type summary
print("\n--- Edge Type Summary ---")
for model in MODELS:
    et_univ = df_edge_types[
        (df_edge_types["model"] == model)
        & (df_edge_types["universality"] == "universal")
        & (~df_edge_types["edge_category"].str.startswith("_"))
    ]
    if len(et_univ) > 0:
        dominant = et_univ.loc[et_univ["fraction"].idxmax()]
        print(
            f"  {model}: dominant edge type = {dominant['edge_category']} ({dominant['fraction']:.1%})"
        )

# Output files
print("\n--- Output Files ---")
for f in (
    sorted(OUT_ANALYSIS.glob("*mechanism*"))
    + sorted(OUT_ANALYSIS.glob("*role*"))
    + sorted(OUT_ANALYSIS.glob("*stability*"))
    + sorted(OUT_ANALYSIS.glob("*layer_mech*"))
):
    print(f"  {f.name}")
for f in sorted(OUT_VIZ.glob("T8_*.png")):
    print(f"  {f.name}")

print("\nDone.")

PHASE 5: MECHANISM PROFILING: SUMMARY

--- Universal Core Dominant Roles ---
  pythia-70m: diffuse (38.9%), then previous_token (30.6%)
  pythia-160m: bos_sink (41.2%), then diffuse (31.2%)
  pythia-410m: bos_sink (49.7%), then diffuse (35.2%)
  pythia-1b: bos_sink (39.7%), then diffuse (36.2%)
  pythia-1.4b: diffuse (42.9%), then bos_sink (39.5%)

--- Enrichment Summary ---
  induction         : mean OR=10.32 (ENRICHED), significant in 3/5 models
  previous_token    : mean OR=8.81 (ENRICHED), significant in 5/5 models
  bos_sink          : mean OR=0.23 (DEPLETED), significant in 5/5 models
  diffuse           : mean OR=2.03 (ENRICHED), significant in 2/5 models

--- Layer Concentration ---
  pythia-70m: peak=middle (56.2% of universal edges)
  pythia-160m: peak=late (47.9% of universal edges)
  pythia-410m: peak=late (50.1% of universal edges)
  pythia-1b: peak=late (43.9% of universal edges)
  pythia-1.4b: peak=middle (51.1% of universal edges)

--- Draw Stability Verdict ---
  pythi